# 01 – Project finance: renewables before and after the Iran war

Project finance deals from LSEG (DSCREEN universe), Jan 2024 to download date. Kept: renewable projects only.
End result: country × technology × month panel with a post-war flag.

Same layout as the bond notebooks:
- `data/raw/pf/` monthly snapshots, one csv per month, never edited
- `data/clean/` `pf_deals.parquet` (one row per deal) and `pf_panel_monthly.parquet`
- `output/pf/` check tables


## Setup

In [ ]:
import re
import warnings
from pathlib import Path

import pandas as pd
import lseg.data as ld

warnings.filterwarnings("ignore", category=FutureWarning)  # lseg spams a pandas downcasting warning
pd.set_option("display.max_columns", 30)

# notebook sits in notebooks/, project root is one up (same as 01_download_bonds)
base_dir = Path.cwd().parent
raw_dir = base_dir / "data" / "raw" / "pf"
clean_dir = base_dir / "data" / "clean"
out_dir = base_dir / "output" / "pf"
for d in [raw_dir, clean_dir, out_dir]:
    d.mkdir(parents=True, exist_ok=True)

start, end = "2024-01-01", pd.Timestamp.today().strftime("%Y-%m-%d")
war_start = pd.Timestamp("2026-02-28")   # US-Israeli strikes, Hormuz closed
refetch_last = 2                          # recent months still get backfilled by LSEG

# None = all countries. LSEG spellings, see the country list in the checks section
countries = None

## Download



In [ ]:
fields = {
    "TR.PJFProjectName":      "project_name",
    "TR.PJFAnnouncementDate": "announced",
    "TR.PJFProjectNation":    "country",
    "TR.PJFProjectRegion":    "region",
    "TR.PJFProjectStatus":    "status",
    "TR.PJFProjectType":      "contract_type",   # BOO, BOT etc, not the technology
    "TR.PJFSponsor":          "sponsor",
    "TR.PJFTotalProjectCost": "cost_usd",
}


def get_month(month):
    month_end = month + pd.offsets.MonthEnd(0)
    universe = ("SCREEN(U(IN(DEALS)), "
                f"BETWEEN(TR.PJFAnnouncementDate,{month:%Y%m%d},{month_end:%Y%m%d}), CURN=USD)")
    df = ld.get_data(universe=universe, fields=list(fields))
    df.columns = ["deal_id"] + list(fields.values())   # comes back in request order
    return df

In [ ]:
ld.open_session()

months = pd.date_range(start, end, freq="MS")
recent = set(months[-refetch_last:])

for month in months:
    f = raw_dir / f"pf_{month:%Y_%m}.csv"
    if f.exists() and month not in recent:
        continue
    try:
        df = get_month(month)
        df.to_csv(f, index=False)
        print(f"{month:%Y-%m}: {len(df):>5} rows")
    except Exception as e:
        print(f"{month:%Y-%m}: failed, {str(e)[:120]}")

ld.close_session()
print(len(list(raw_dir.glob("pf_*.csv"))), "monthly files in", raw_dir)

## Tidy



In [ ]:
raw = pd.concat([pd.read_csv(f) for f in sorted(raw_dir.glob("pf_*.csv"))], ignore_index=True)

raw["announced"] = pd.to_datetime(raw["announced"], errors="coerce")
raw["cost_usd"] = pd.to_numeric(raw["cost_usd"], errors="coerce")
raw.loc[raw["cost_usd"] <= 0, "cost_usd"] = pd.NA
for col in ["project_name", "country", "region", "status", "contract_type", "sponsor"]:
    raw[col] = raw[col].astype("string").str.strip().replace("", pd.NA)


def join_unique(s):
    names = sorted(set(s.dropna()))
    return "; ".join(names) if names else pd.NA


deals = raw.groupby("deal_id", as_index=False).agg(
    project_name=("project_name", "first"),
    announced=("announced", "first"),
    country=("country", "first"),
    region=("region", "first"),
    status=("status", "first"),
    contract_type=("contract_type", "first"),
    sponsors=("sponsor", join_unique),
    cost_usd=("cost_usd", "max"),
)

print(f"{len(raw):,} rows -> {len(deals):,} deals, cost disclosed for {deals['cost_usd'].notna().mean():.0%}")

## Technology from project name



In [ ]:
tech_rules = [
    ("Hydrogen",        r"hydrogen|ammonia|e-?fuel|electrolys"),
    ("Solar + storage", r"(solar|\bpv\b|photovoltaic).*(storage|battery|\bbess\b)|(storage|battery|\bbess\b).*(solar|\bpv\b)"),
    ("Solar",           r"solar|\bpv\b|photovoltaic|agrivoltaic"),
    ("Offshore wind",   r"offshore wind"),
    ("Onshore wind",    r"\bwind\b|wind ?farm|wind ?park|wind ?power"),
    ("Hydro",           r"\bhydro\b|hydropower|hydroelectric|hydel|pumped storage|\bdam\b"),
    ("Storage",         r"battery|\bbess\b|energy storage"),
    ("Geothermal",      r"geothermal"),
    ("Bioenergy",       r"biomass|biogas|biomethane|bioenergy|biofuel|waste[- ]to[- ]energy|\bwte\b"),
]
renewables = {"Solar + storage", "Solar", "Offshore wind", "Onshore wind",
              "Hydro", "Storage", "Geothermal", "Bioenergy"}


def tech_from_name(name):
    if pd.isna(name):
        return "Other"
    name = name.lower()
    for tech, pattern in tech_rules:
        if re.search(pattern, name):
            return tech
    return "Other"


deals["technology"] = deals["project_name"].apply(tech_from_name)
deals["renewable"] = deals["technology"].isin(renewables)
deals["technology"].value_counts()

In [ ]:
# renewables hiding in Other? add keywords above if so
other = deals.loc[deals["technology"] == "Other", ["project_name", "country"]]
other.sample(min(25, len(other)), random_state=1)

## War timing


In [ ]:
war_month = war_start.to_period("M")
deals["month"] = deals["announced"].dt.to_period("M")
deals["post_war"] = (deals["announced"] >= war_start).astype(int)
deals["event_month"] = deals["month"].apply(lambda m: (m - war_month).n)

ren = deals[deals["renewable"]].copy()
if countries is not None:
    ren = ren[ren["country"].isin(countries)]

print(f"{len(ren):,} renewable deals")
ren["post_war"].value_counts().rename({0: "pre", 1: "post"})

## Checks


In [ ]:
monthly_counts = deals.groupby("month").agg(all_deals=("deal_id", "count"),
                                            renewable=("renewable", "sum"))
monthly_counts.to_csv(out_dir / "monthly_counts.csv")
display(monthly_counts)

for m in monthly_counts["all_deals"].nlargest(3).index:
    in_m = deals[deals["month"] == m]
    print(f"\n{m}: {len(in_m)} deals")
    print("  countries:", in_m["country"].value_counts().head(5).to_dict())
    print("  status:   ", in_m["status"].value_counts().head(5).to_dict())

In [ ]:
# LSEG country spellings, for the countries list in Setup
ren["country"].value_counts().head(40)

## Panel: country × technology × month



In [ ]:
panel = ren.groupby(["country", "technology", "month"]).agg(
    n_deals=("deal_id", "count"),
    n_with_cost=("cost_usd", "count"),
    cost_usd_bn=("cost_usd", lambda x: x.sum() / 1e9),
)

all_months = pd.period_range(start, end, freq="M")
pairs = panel.reset_index()[["country", "technology"]].drop_duplicates()
full = pd.MultiIndex.from_tuples(
    [(c, t, m) for c, t in pairs.itertuples(index=False) for m in all_months],
    names=["country", "technology", "month"],
)
panel = panel.reindex(full, fill_value=0).reset_index()

panel["post_war"] = (panel["month"] >= war_month).astype(int)
panel["event_month"] = panel["month"].apply(lambda m: (m - war_month).n)

print(panel.shape)
panel.head()

In [ ]:
# parquet doesn't take Period, store month as timestamp (first of month)
to_save = lambda df: df.assign(month=df["month"].dt.to_timestamp())

to_save(ren).to_parquet(clean_dir / "pf_deals.parquet", index=False)
to_save(panel).to_parquet(clean_dir / "pf_panel_monthly.parquet", index=False)
print("saved pf_deals.parquet and pf_panel_monthly.parquet in", clean_dir)